# Semiconductor Patent RAG — Clean End-to-End Baseline

This notebook consolidates the two original notebooks into one reproducible pipeline:

**500 patent corpus → audit → parse → chunk → Ollama embeddings → semantic retrieval → grounded RAG answer → evaluation**

The patent text files on disk are treated as the authoritative corpus. Stale/incomplete metadata is not used as the source of truth.

In [1]:
from pathlib import Path
import re
import html
import time
import requests
import numpy as np
import pandas as pd
import json

PROJECT_ROOT = Path.cwd()
CORPUS_ROOT = PROJECT_ROOT / "semiconductor_patents"

CATEGORIES = [
    "semiconductor_manufacturing",
    "transistor_device_technology",
    "memory",
    "advanced_packaging",
    "power_semiconductors",
    "photonics",
    "ai_advanced_semiconductor",
]

EMBED_MODEL = "nomic-embed-text"
LLM_MODEL = "qwen3:8b"
OLLAMA_BASE_URL = "http://localhost:11434"

CHUNK_SIZE = 1500
CHUNK_OVERLAP = 250
TOP_K = 5
CANDIDATE_K = 30

print("Project root:", PROJECT_ROOT)
print("Corpus root :", CORPUS_ROOT)
print("Embedding   :", EMBED_MODEL)
print("LLM         :", LLM_MODEL)

Project root: /Users/jamesjr/Documents/SemiconductorPatentRAG
Corpus root : /Users/jamesjr/Documents/SemiconductorPatentRAG/semiconductor_patents
Embedding   : nomic-embed-text
LLM         : qwen3:8b


## 1. Audit the corpus

The original collection produced 500 text documents in seven categories. We verify the actual files before parsing anything.

In [2]:
def list_patent_files():
    files = []
    for category in CATEGORIES:
        directory = CORPUS_ROOT / category
        if directory.exists():
            files.extend(sorted(directory.glob("*.txt")))
    return files

patent_files = list_patent_files()

inventory = pd.DataFrame([
    {
        "category": category,
        "files": sum(p.parent.name == category for p in patent_files),
    }
    for category in CATEGORIES
])

display(inventory)
print("Total patent files:", len(patent_files))
assert len(patent_files) == 500, f"Expected 500 files, found {len(patent_files)}"

,category,files
0,semiconductor_manufacturing,100
1,transistor_device_technology,100
2,memory,75
3,advanced_packaging,75
4,power_semiconductors,50
5,photonics,50
6,ai_advanced_semiconductor,50


Total patent files: 500


## 2. Parse and normalize patents

The original notebooks contained several competing parsers. This version uses one parser and handles the two header spellings that appeared in the corpus.

In [3]:
def clean_text(text):
    text = "" if text is None else str(text)
    text = html.unescape(text)
    text = text.replace("\x00", " ")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def header_value(text, field):
    match = re.search(
        rf"(?im)^\s*{re.escape(field)}:\s*(.+?)\s*$",
        text,
    )
    return match.group(1).strip() if match else ""

def section_value(text, starts, ends):
    start = None
    for pattern in starts:
        match = re.search(pattern, text, re.I | re.M)
        if match:
            start = match.end()
            break
    if start is None:
        return ""

    remaining = text[start:]
    positions = []
    for pattern in ends:
        match = re.search(pattern, remaining, re.I | re.M)
        if match:
            positions.append(match.start())

    if positions:
        remaining = remaining[:min(positions)]
    return clean_text(remaining)

def parse_patent(path):
    raw = clean_text(path.read_text(encoding="utf-8", errors="replace"))

    document_id = (
        header_value(raw, "DOCUMENT ID")
        or header_value(raw, "DOCUMENT_ID")
        or path.stem.removeprefix("patent_")
    )

    category = header_value(raw, "CATEGORY") or path.parent.name
    cpc_section = header_value(raw, "CPC SECTION")
    source = header_value(raw, "SOURCE") or "BIGPATENT"

    abstract = section_value(
        raw,
        [r"^\s*ABSTRACT\s*=*\s*$"],
        [
            r"^\s*DETAILED DESCRIPTION\s*=*\s*$",
            r"^\s*DESCRIPTION\s*=*\s*$",
            r"^\s*CLAIMS\s*=*\s*$",
        ],
    )

    description = section_value(
        raw,
        [
            r"^\s*DETAILED DESCRIPTION\s*=*\s*$",
            r"^\s*DESCRIPTION\s*=*\s*$",
        ],
        [
            r"^\s*CLAIMS\s*=*\s*$",
            r"^\s*REFERENCES CITED\s*$",
        ],
    )

    claims = section_value(
        raw,
        [r"^\s*CLAIMS\s*=*\s*$"],
        [
            r"^\s*REFERENCES CITED\s*$",
            r"^\s*REFERENCE\s*$",
        ],
    )

    return {
        "document_id": document_id,
        "category": category,
        "cpc_section": cpc_section,
        "source": source,
        "abstract": abstract,
        "description": description,
        "claims": claims,
        "filename": path.name,
        "file_path": str(path),
        "word_count": len(re.findall(r"\b\w+\b", raw)),
        "character_count": len(raw),
    }

patents_df = pd.DataFrame(parse_patent(p) for p in patent_files)

print("Records:", len(patents_df))
print("Unique IDs:", patents_df.document_id.nunique())
print("Missing IDs:", patents_df.document_id.eq("").sum())
print("Missing abstracts:", patents_df.abstract.eq("").sum())
print("Missing descriptions:", patents_df.description.eq("").sum())
print("Claims available:", patents_df.claims.ne("").sum())

assert len(patents_df) == 500
assert patents_df.document_id.ne("").all()
assert patents_df.document_id.is_unique
assert patents_df.description.ne("").all()

Records: 500
Unique IDs: 500
Missing IDs: 0
Missing abstracts: 0
Missing descriptions: 0
Claims available: 0


In [4]:
display(
    patents_df.groupby("category")
    .agg(
        documents=("document_id", "count"),
        avg_description_words=("description", lambda s: s.str.split().str.len().mean()),
        median_description_words=("description", lambda s: s.str.split().str.len().median()),
    )
    .sort_index()
)

print("\nRepresentative patent:")
display(patents_df.head(1).T)

,documents,avg_description_words,median_description_words
category,,,
advanced_packaging,75,4879.120000,4159.0
ai_advanced_semiconductor,50,5920.600000,5392.5
memory,75,6084.706667,5452.0
photonics,50,5542.560000,5023.0
power_semiconductors,50,4398.360000,3903.5
semiconductor_manufacturing,100,5649.000000,4672.0
transistor_device_technology,100,4824.100000,4525.0



Representative patent:


,0
document_id,035ed7683a8cf2c3
category,semiconductor_manufacturing
cpc_section,
source,BIGPATENT
abstract,A method for semiconductor device feature deve...
description,FIELD OF THE INVENTION [0001] This invention g...
claims,
filename,patent_035ed7683a8cf2c3.txt
file_path,/Users/jamesjr/Documents/SemiconductorPatentRA...
word_count,3538


## 3. Section-aware chunking

Abstracts are kept as their own chunks. Descriptions are split on apparent uppercase section headings and then into overlapping character windows.

In [5]:
HEADING_PATTERN = re.compile(
    r"(?im)^\s*([A-Z][A-Z0-9 ,/&()\-]{3,80})\s*$"
)

IGNORED_HEADINGS = {"ABSTRACT", "DESCRIPTION", "DETAILED DESCRIPTION"}

def split_sections(text):
    matches = list(HEADING_PATTERN.finditer(text))
    if not matches:
        return [("DESCRIPTION", clean_text(text))]

    sections = []
    for i, match in enumerate(matches):
        heading = clean_text(match.group(1))
        if heading in IGNORED_HEADINGS:
            continue

        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        value = clean_text(text[start:end])

        if value:
            sections.append((heading, value))

    return sections or [("DESCRIPTION", clean_text(text))]

def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    if not text:
        return []
    if overlap >= size:
        raise ValueError("CHUNK_OVERLAP must be smaller than CHUNK_SIZE")

    chunks = []
    step = size - overlap
    start = 0

    while start < len(text):
        end = min(start + size, len(text))
        value = text[start:end].strip()
        if value:
            chunks.append(value)
        if end >= len(text):
            break
        start += step

    return chunks

chunk_records = []

for _, patent in patents_df.iterrows():
    if patent["abstract"]:
        chunk_records.append({
            "document_id": patent["document_id"],
            "category": patent["category"],
            "cpc_section": patent["cpc_section"],
            "source": patent["source"],
            "section": "ABSTRACT",
            "chunk_index": 0,
            "text": patent["abstract"],
            "file_path": patent["file_path"],
        })

    index = 1
    for section_name, section_text in split_sections(patent["description"]):
        for part, value in enumerate(chunk_text(section_text), start=1):
            chunk_records.append({
                "document_id": patent["document_id"],
                "category": patent["category"],
                "cpc_section": patent["cpc_section"],
                "source": patent["source"],
                "section": section_name,
                "chunk_index": index,
                "part": part,
                "text": value,
                "file_path": patent["file_path"],
            })
            index += 1

chunks_df = pd.DataFrame(chunk_records)
chunks_df["chunk_id"] = (
    chunks_df.document_id.astype(str) + "_" +
    chunks_df.chunk_index.astype(str)
)
chunks_df["word_count"] = chunks_df.text.str.split().str.len()

print("Documents:", chunks_df.document_id.nunique())
print("Chunks:", len(chunks_df))
print("Average words/chunk:", round(chunks_df.word_count.mean(), 1))
print("Empty chunks:", chunks_df.text.eq("").sum())

assert chunks_df.document_id.nunique() == 500
assert chunks_df.chunk_id.is_unique
assert chunks_df.text.ne("").all()

display(chunks_df[["document_id", "category", "section", "word_count"]].head())

Documents: 500
Chunks: 13220
Average words/chunk: 239.6
Empty chunks: 0


,document_id,category,section,word_count
0,035ed7683a8cf2c3,semiconductor_manufacturing,ABSTRACT,104
1,035ed7683a8cf2c3,semiconductor_manufacturing,DESCRIPTION,205
2,035ed7683a8cf2c3,semiconductor_manufacturing,DESCRIPTION,207
3,035ed7683a8cf2c3,semiconductor_manufacturing,DESCRIPTION,230
4,035ed7683a8cf2c3,semiconductor_manufacturing,DESCRIPTION,233


## 4. Ollama embedding and generation

The original notebooks tested BGE embeddings and later switched to `nomic-embed-text`. The clean baseline uses one embedding model consistently for both corpus and query embeddings.

In [6]:
OLLAMA_EMBED_URL = f"{OLLAMA_BASE_URL}/api/embeddings"
OLLAMA_GENERATE_URL = f"{OLLAMA_BASE_URL}/api/generate"

def ollama_json(url, payload, timeout=120):
    response = requests.post(url, json=payload, timeout=timeout)
    response.raise_for_status()
    return response.json()

def embed_text(text):
    result = ollama_json(
        OLLAMA_EMBED_URL,
        {"model": EMBED_MODEL, "prompt": text},
    )
    return np.asarray(result["embedding"], dtype=np.float32)

def ask_ollama(prompt):
    result = ollama_json(
        OLLAMA_GENERATE_URL,
        {
            "model": LLM_MODEL,
            "prompt": prompt,
            "stream": False,
        },
        timeout=180,
    )
    return result["response"]

tags = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=10)
tags.raise_for_status()
available_models = {m["name"] for m in tags.json().get("models", [])}

print("Ollama connection: OK")
print("Embedding model available:", EMBED_MODEL in available_models)
print("LLM model available:", LLM_MODEL in available_models)

Ollama connection: OK
Embedding model available: False
LLM model available: True


In [7]:
test_embedding = embed_text(chunks_df.iloc[0]["text"])

print("Embedding dimension:", test_embedding.shape[0])
print("Embedding dtype:", test_embedding.dtype)
print("Embedding norm:", round(float(np.linalg.norm(test_embedding)), 4))

assert test_embedding.ndim == 1
assert test_embedding.shape[0] > 0

Embedding dimension: 768
Embedding dtype: float32
Embedding norm: 19.8335


## 5. Embed the corpus

For this 500-document baseline, vectors are held in memory. A production version can persist the same records and vectors in a vector index.

In [8]:
vectors = []
start = time.time()

for i, text in enumerate(chunks_df.text, start=1):
    vectors.append(embed_text(text))
    if i % 100 == 0 or i == len(chunks_df):
        print(f"Embedded {i:,}/{len(chunks_df):,}")

embedding_matrix = np.asarray(vectors, dtype=np.float32)
embedding_matrix = embedding_matrix / np.maximum(
    np.linalg.norm(embedding_matrix, axis=1, keepdims=True),
    1e-12,
)

print("Shape:", embedding_matrix.shape)
print(f"Elapsed: {(time.time() - start) / 60:.1f} minutes")

Embedded 100/13,220
Embedded 200/13,220
Embedded 300/13,220
Embedded 400/13,220
Embedded 500/13,220
Embedded 600/13,220
Embedded 700/13,220
Embedded 800/13,220
Embedded 900/13,220
Embedded 1,000/13,220
Embedded 1,100/13,220
Embedded 1,200/13,220
Embedded 1,300/13,220
Embedded 1,400/13,220
Embedded 1,500/13,220
Embedded 1,600/13,220
Embedded 1,700/13,220
Embedded 1,800/13,220
Embedded 1,900/13,220
Embedded 2,000/13,220
Embedded 2,100/13,220
Embedded 2,200/13,220
Embedded 2,300/13,220
Embedded 2,400/13,220
Embedded 2,500/13,220
Embedded 2,600/13,220
Embedded 2,700/13,220
Embedded 2,800/13,220
Embedded 2,900/13,220
Embedded 3,000/13,220
Embedded 3,100/13,220
Embedded 3,200/13,220
Embedded 3,300/13,220
Embedded 3,400/13,220
Embedded 3,500/13,220
Embedded 3,600/13,220
Embedded 3,700/13,220
Embedded 3,800/13,220
Embedded 3,900/13,220
Embedded 4,000/13,220
Embedded 4,100/13,220
Embedded 4,200/13,220
Embedded 4,300/13,220
Embedded 4,400/13,220
Embedded 4,500/13,220
Embedded 4,600/13,220
Embedd

In [19]:
import faiss
import numpy as np
from pathlib import Path

# -------------------------------------------------
# 1. Create & save FAISS index
# -------------------------------------------------
dim = embedding_matrix.shape[1]
index = faiss.IndexFlatIP(dim)          # Inner Product (vectors are already L2-normalized)
index.add(embedding_matrix)

faiss.write_index(index, "embeddings.faiss")

# -------------------------------------------------
# 2. Save ALL columns from chunks_df
# -------------------------------------------------
chunks_df.to_parquet("chunks_meta.parquet", index=False)

# Optional: also keep the raw matrix
np.save("embedding_matrix.npy", embedding_matrix)

print(f"Saved FAISS index with {index.ntotal:,} vectors")
print(f"Saved full metadata ({chunks_df.shape[1]} columns) for {len(chunks_df):,} chunks")

Saved FAISS index with 13,220 vectors
Saved full metadata (11 columns) for 13,220 chunks
